# GNSS-Denied Navigation — Satellite-to-Drone Cross-View Matching

Train a feature extractor to match drone downward camera views against satellite/aerial reference imagery.
Enables autonomous position estimation without GPS.

**Approach:** Contrastive learning (triplet loss) — learn embeddings where same-location drone+satellite pairs are close, different locations are far apart.

**Dataset:** University-1652 (drone ↔ satellite cross-view pairs) + custom ESRI tiles

**Result:** Feature extractor that can localize a drone image against a pre-loaded satellite map.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/DroneCV/gnss_denied'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/models', exist_ok=True)
os.makedirs(f'{WORK_DIR}/data', exist_ok=True)

In [ ]:
!pip install -q torch torchvision timm gdown

# Generate training data from ESRI World Imagery tiles
# Creates drone↔satellite pairs by cropping/augmenting tiles

import requests, math, os, cv2
import numpy as np

DATA_DIR = "/content/data/crossview_pairs"
os.makedirs(f"{DATA_DIR}/satellite", exist_ok=True)
os.makedirs(f"{DATA_DIR}/drone", exist_ok=True)

np.random.seed(42)

# Sample locations across Europe for training diversity
base_coords = [
    (60.17, 24.94), (60.13, 24.51), (60.19, 25.02), (60.22, 24.87),
    (59.33, 18.07), (55.68, 12.57), (52.52, 13.41), (48.86, 2.35),
    (51.51, -0.13), (40.42, -3.70), (41.39, 2.17), (45.46, 9.19),
]
locations = []
for lat, lon in base_coords:
    for _ in range(17):
        locations.append((lat + np.random.uniform(-0.02, 0.02),
                          lon + np.random.uniform(-0.03, 0.03)))

def lat_lon_to_tile(lat, lon, zoom):
    n = 2**zoom
    x = int((lon + 180) / 360 * n)
    y = int((1 - math.log(math.tan(math.radians(lat)) + 1/math.cos(math.radians(lat))) / math.pi) / 2 * n)
    return x, y

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (research use)"})
success = 0

print(f"Downloading {len(locations)} satellite tiles from ESRI World Imagery...")
for i, (lat, lon) in enumerate(locations):
    x, y = lat_lon_to_tile(lat, lon, 18)
    url = f"https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/18/{y}/{x}"
    try:
        r = session.get(url, timeout=15)
        if r.status_code == 200 and len(r.content) > 5000:
            sat_path = f"{DATA_DIR}/satellite/{i:04d}.jpg"
            with open(sat_path, "wb") as f:
                f.write(r.content)
            # Decode and create augmented drone view
            sat_img = cv2.imdecode(np.frombuffer(r.content, np.uint8), cv2.IMREAD_COLOR)
            if sat_img is None: continue
            h, w = sat_img.shape[:2]
            crop = int(w * np.random.uniform(0.5, 0.9))
            cx = np.random.randint(0, max(1, w - crop))
            cy = np.random.randint(0, max(1, h - crop))
            drone_view = sat_img[cy:cy+crop, cx:cx+crop]
            drone_view = cv2.resize(drone_view, (256, 256))
            angle = np.random.uniform(-30, 30)
            M = cv2.getRotationMatrix2D((128, 128), angle, 1.0)
            drone_view = cv2.warpAffine(drone_view, M, (256, 256))
            drone_view = cv2.GaussianBlur(drone_view, (3, 3), 0)
            drone_view = np.clip(drone_view.astype(float) * np.random.uniform(0.8, 1.2) + np.random.uniform(-20, 20), 0, 255).astype(np.uint8)
            cv2.imwrite(f"{DATA_DIR}/drone/{i:04d}.jpg", drone_view)
            success += 1
        elif i == 0:
            print(f"  First tile status: {r.status_code}, size: {len(r.content)}")
    except Exception as e:
        if i == 0: print(f"  Error on first tile: {e}")
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(locations)} attempted, {success} successful")

print(f"
Done: {success} pairs generated")
if success == 0:
    print("ESRI tiles blocked. Falling back to OpenStreetMap static maps...")
    for i, (lat, lon) in enumerate(locations[:100]):
        # Use OSM static map API as fallback
        url = f"https://static-maps.yandex.ru/v1?ll={lon},{lat}&z=17&size=256,256&l=sat"
        try:
            r = session.get(url, timeout=10)
            if r.status_code == 200 and len(r.content) > 3000:
                with open(f"{DATA_DIR}/satellite/{i:04d}.jpg", "wb") as f:
                    f.write(r.content)
                sat_img = cv2.imdecode(np.frombuffer(r.content, np.uint8), cv2.IMREAD_COLOR)
                if sat_img is not None:
                    drone_view = cv2.resize(sat_img, (256, 256))
                    angle = np.random.uniform(-30, 30)
                    M = cv2.getRotationMatrix2D((128, 128), angle, 1.0)
                    drone_view = cv2.warpAffine(drone_view, M, (256, 256))
                    cv2.imwrite(f"{DATA_DIR}/drone/{i:04d}.jpg", drone_view)
                    success += 1
        except: pass
    print(f"Fallback: {success} pairs")

In [ ]:
import glob, cv2
import matplotlib.pyplot as plt

sat_images = sorted(glob.glob("/content/data/crossview_pairs/satellite/*.jpg"))
drone_images = sorted(glob.glob("/content/data/crossview_pairs/drone/*.jpg"))
print(f"Training pairs: {len(drone_images)} drone ↔ {len(sat_images)} satellite")

if len(sat_images) >= 5:
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for i in range(5):
        idx = i * (len(sat_images) // 5)
        sat = cv2.cvtColor(cv2.imread(sat_images[idx]), cv2.COLOR_BGR2RGB)
        drone = cv2.cvtColor(cv2.imread(drone_images[idx]), cv2.COLOR_BGR2RGB)
        axes[0, i].imshow(sat); axes[0, i].set_title(f"Sat {idx}"); axes[0, i].axis("off")
        axes[1, i].imshow(drone); axes[1, i].set_title(f"Drone {idx}"); axes[1, i].axis("off")
    plt.suptitle("Training Pairs: Satellite (top) vs Simulated Drone (bottom)")
    plt.tight_layout(); plt.show()
else:
    print("ERROR: No tiles downloaded. Check network/ESRI access.")
    print("Trying alternative: generating from random noise for structure test...")
    import numpy as np, os
    os.makedirs("/content/data/crossview_pairs/satellite", exist_ok=True)
    os.makedirs("/content/data/crossview_pairs/drone", exist_ok=True)
    for i in range(100):
        img = np.random.randint(50, 200, (256,256,3), dtype=np.uint8)
        cv2.imwrite(f"/content/data/crossview_pairs/satellite/{i:04d}.jpg", img)
        aug = cv2.GaussianBlur(img, (5,5), 0)
        cv2.imwrite(f"/content/data/crossview_pairs/drone/{i:04d}.jpg", aug)
    print("Generated 100 synthetic pairs for pipeline validation")

In [ ]:
import glob

# Check dataset structure
train_drone = sorted(glob.glob('/content/data/crossview_pairs/drone/*.jpg'))
train_sat = sorted(glob.glob('/content/data/crossview_pairs/satellite/*.jpg'))
print(f'Training: {len(train_drone)} drone images, {len(train_sat)} satellite images')
print(f'Locations: {len(set(os.path.basename(os.path.dirname(p)) for p in train_drone))}')


from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import random

class CrossViewDataset(Dataset):
    """Triplet dataset: (drone, satellite_pos, satellite_neg)."""
    def __init__(self, data_dir, img_size=256):
        self.sat_imgs = sorted(glob.glob(f"{data_dir}/satellite/*.jpg"))
        self.drone_imgs = sorted(glob.glob(f"{data_dir}/drone/*.jpg"))
        assert len(self.sat_imgs) == len(self.drone_imgs), "Mismatch"
        self.n = len(self.sat_imgs)
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        print(f"Dataset: {self.n} pairs")

    def __len__(self):
        return self.n * 5  # 5 triplets per pair per epoch

    def __getitem__(self, idx):
        i = idx % self.n
        drone = Image.open(self.drone_imgs[i]).convert("RGB")
        sat_pos = Image.open(self.sat_imgs[i]).convert("RGB")
        # Negative: random different location
        neg_idx = random.choice([j for j in range(self.n) if j != i])
        sat_neg = Image.open(self.sat_imgs[neg_idx]).convert("RGB")
        return self.transform(drone), self.transform(sat_pos), self.transform(sat_neg)

train_ds = CrossViewDataset("/content/data/crossview_pairs")
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
import torch
import torch.nn as nn
import timm

class CrossViewEncoder(nn.Module):
    """Shared encoder for drone and satellite views."""
    def __init__(self, backbone='resnet50', embed_dim=512):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=True, num_classes=0)
        feat_dim = self.backbone.num_features
        self.embed = nn.Sequential(
            nn.Linear(feat_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.embed(features)
        return nn.functional.normalize(embedding, p=2, dim=1)

model = CrossViewEncoder('resnet50', embed_dim=512).cuda()
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Step 3: Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import random, glob

class CrossViewDataset(Dataset):
    """Triplet dataset: (drone, satellite_pos, satellite_neg)."""
    def __init__(self, data_dir, img_size=256):
        self.sat_imgs = sorted(glob.glob(f"{data_dir}/satellite/*.jpg"))
        self.drone_imgs = sorted(glob.glob(f"{data_dir}/drone/*.jpg"))
        self.n = min(len(self.sat_imgs), len(self.drone_imgs))
        assert self.n > 0, f"No images found in {data_dir}"
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        print(f"Dataset: {self.n} pairs")

    def __len__(self):
        return self.n * 5

    def __getitem__(self, idx):
        i = idx % self.n
        drone = Image.open(self.drone_imgs[i]).convert("RGB")
        sat_pos = Image.open(self.sat_imgs[i]).convert("RGB")
        neg_idx = random.choice([j for j in range(self.n) if j != i])
        sat_neg = Image.open(self.sat_imgs[neg_idx]).convert("RGB")
        return self.transform(drone), self.transform(sat_pos), self.transform(sat_neg)

train_ds = CrossViewDataset("/content/data/crossview_pairs")
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
print(f"Batches per epoch: {len(train_loader)}")


## Step 4: Training (Triplet Loss)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
triplet_loss = nn.TripletMarginLoss(margin=0.3)

EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_idx, (drone, sat_pos, sat_neg) in enumerate(train_loader):
        drone, sat_pos, sat_neg = drone.cuda(), sat_pos.cuda(), sat_neg.cuda()

        # Forward
        emb_drone = model(drone)
        emb_pos = model(sat_pos)
        emb_neg = model(sat_neg)

        loss = triplet_loss(emb_drone, emb_pos, emb_neg)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{EPOCHS} — Loss: {avg_loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}')

# Save model
torch.save(model.state_dict(), f'{WORK_DIR}/models/crossview_resnet50_512d.pth')
print(f'Model saved to {WORK_DIR}/models/crossview_resnet50_512d.pth')

## Step 5: Evaluate — Retrieval Accuracy

Given a drone image, find the correct satellite tile (top-1, top-5 accuracy).

In [ ]:
import numpy as np

model.eval()

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Build satellite gallery embeddings
test_sat = sorted(glob.glob('/content/data/crossview_pairs/satellite/*.jpg'))
test_drone = sorted(glob.glob('/content/data/crossview_pairs/drone/*.jpg'))

def embed_images(paths, model, transform, batch_size=64):
    embeddings, labels = [], []
    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i:i+batch_size]
        imgs = torch.stack([transform(Image.open(p).convert('RGB')) for p in batch_paths]).cuda()
        with torch.no_grad():
            emb = model(imgs)
        embeddings.append(emb.cpu().numpy())
        labels.extend([os.path.basename(os.path.dirname(p)) for p in batch_paths])
    return np.vstack(embeddings), labels

print('Embedding satellite gallery...')
sat_emb, sat_labels = embed_images(test_sat, model, eval_transform)
print(f'Gallery: {sat_emb.shape}')

print('Embedding drone queries...')
drone_emb, drone_labels = embed_images(test_drone[:500], model, eval_transform)
print(f'Queries: {drone_emb.shape}')

# Compute retrieval accuracy
distances = np.dot(drone_emb, sat_emb.T)  # cosine similarity (embeddings are normalized)
top1, top5, top10 = 0, 0, 0
for i, query_label in enumerate(drone_labels):
    ranked = np.argsort(-distances[i])
    retrieved_labels = [sat_labels[r] for r in ranked]
    if query_label in retrieved_labels[:1]: top1 += 1
    if query_label in retrieved_labels[:5]: top5 += 1
    if query_label in retrieved_labels[:10]: top10 += 1

n = len(drone_labels)
print(f'\nRetrieval Accuracy:')
print(f'  Top-1:  {top1/n*100:.1f}%')
print(f'  Top-5:  {top5/n*100:.1f}%')
print(f'  Top-10: {top10/n*100:.1f}%')


## Step 6: Test on Custom Satellite Tiles

Load our own ESRI tiles and test matching against a simulated drone view.

In [ ]:
# Upload your satellite tiles to Drive, or clone from repo:
# !git clone https://github.com/rwiren/drone-cv-detection.git /content/repo
# Then copy: data/satellite_tiles/jorvas_satellite_z18_stitched.jpg

import cv2

# For demo: create tiles from stitched reference
ref_path = f'{WORK_DIR}/data/jorvas_satellite_z18_stitched.jpg'
if os.path.exists(ref_path):
    ref = cv2.imread(ref_path)
    # Split into 3x3 grid of 256x256 tiles
    tile_embeddings = []
    tile_centers = []
    tile_size = 256
    for row in range(3):
        for col in range(3):
            tile = ref[row*tile_size:(row+1)*tile_size, col*tile_size:(col+1)*tile_size]
            tile_rgb = cv2.cvtColor(tile, cv2.COLOR_BGR2RGB)
            tile_tensor = eval_transform(Image.fromarray(tile_rgb)).unsqueeze(0).cuda()
            with torch.no_grad():
                emb = model(tile_tensor)
            tile_embeddings.append(emb.cpu().numpy().flatten())
            tile_centers.append((col * tile_size + tile_size//2, row * tile_size + tile_size//2))

    tile_embeddings = np.array(tile_embeddings)
    print(f'Embedded {len(tile_embeddings)} reference tiles')

    # Simulate drone view (crop from reference + augment)
    drone_crop = ref[200:456, 200:456]  # 256x256 crop
    drone_crop = cv2.GaussianBlur(drone_crop, (3,3), 0)
    drone_rgb = cv2.cvtColor(drone_crop, cv2.COLOR_BGR2RGB)
    drone_tensor = eval_transform(Image.fromarray(drone_rgb)).unsqueeze(0).cuda()
    with torch.no_grad():
        drone_emb = model(drone_tensor).cpu().numpy().flatten()

    # Find best matching tile
    similarities = np.dot(tile_embeddings, drone_emb)
    best_idx = np.argmax(similarities)
    print(f'Best match: tile {best_idx}, center={tile_centers[best_idx]}, similarity={similarities[best_idx]:.3f}')
    print(f'True position: (328, 328) — error: {np.sqrt((tile_centers[best_idx][0]-328)**2 + (tile_centers[best_idx][1]-328)**2) * 0.30:.1f}m')
else:
    print(f'Upload {ref_path} to test on custom tiles')

## Summary

**What we trained:** A ResNet50-based cross-view feature extractor (512-d embeddings).

**How it's used for GNSS-denied flight:**
1. Pre-flight: embed all satellite tiles of the flight area → build gallery
2. In-flight: embed downward camera frame → find nearest satellite tile → get position
3. Send position to ArduPilot via `VISION_POSITION_ESTIMATE`

**Integration:** The trained model runs in `gnss_denied_nav.py` on the ground server,
receiving the drone's downward camera RTSP stream over 5G.

**Next steps:**
- Fine-tune on Finnish NLS orthophotos for local terrain
- Add rotation invariance (drone heading unknown without GPS)
- Test at multiple altitudes (10m, 30m, 50m, 100m)